# Hospital Generalizability , Set A vs Set B

Testing whether XGBoost B, the best performing model, generalises consistently across the two hospital systems in the dataset.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from src.config import RANDOM_SEED, DATA_DIR, MODELS_DIR, FIGURES_DIR, METRICS_DIR, ALL_FEATURES
from src.data_loader import load_all_patients
from src.preprocessing import engineer_labels, clip_outliers
from src.features import add_lag_features
from src.utils import set_all_seeds, create_patient_splits
from src.evaluate import bootstrap_ci, compute_all_metrics

set_all_seeds(RANDOM_SEED)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
print('Imports OK')

In [ ]:
df = load_all_patients(DATA_DIR)
df, _ = engineer_labels(df)
df    = clip_outliers(df)

patient_train, patient_val, patient_test = create_patient_splits(df)
test_df = df[df['patient_id'].isin(patient_test)].copy()

# Split test set by hospital
test_A = test_df[test_df['hospital_id'] == 'A'].copy()
test_B = test_df[test_df['hospital_id'] == 'B'].copy()

print(f'Test Set A: {test_A["patient_id"].nunique():,} patients | '
      f'prevalence: {test_A["EarlyLabel"].mean()*100:.2f}%')
print(f'Test Set B: {test_B["patient_id"].nunique():,} patients | '
      f'prevalence: {test_B["EarlyLabel"].mean()*100:.2f}%')

In [ ]:
# Load XGBoost B artifacts
meta_cols   = {'patient_id', 'hospital_id', 'timestep', 'SepsisLabel', 'EarlyLabel'}
original_feats = [c for c in ALL_FEATURES if c in test_df.columns]

feat_B = joblib.load(f'{MODELS_DIR}strategy_B_lag_feature_names.pkl')
imp_B  = joblib.load(f'{MODELS_DIR}strategy_B_lag_imputer.pkl')
scl_B  = joblib.load(f'{MODELS_DIR}strategy_B_lag_scaler.pkl')
xgb_B  = joblib.load(f'{MODELS_DIR}xgb_condition_B.pkl')

def preprocess_B(df):
    df = add_lag_features(df)
    df = df.copy()
    all_feat_cols = [c for c in df.columns if c not in meta_cols]
    for feat in original_feats:
        if df[feat].isna().any():
            df[f'{feat}_missing'] = df[feat].isna().astype('int8')
    df[all_feat_cols] = df.groupby('patient_id')[all_feat_cols].ffill()
    for col in feat_B:
        if col not in df.columns:
            df[col] = 0
    X = scl_B.transform(imp_B.transform(df[feat_B].values))
    y = df['EarlyLabel'].values
    return X, y

X_A, y_A = preprocess_B(test_A)
X_B, y_B = preprocess_B(test_B)

probs_A = xgb_B.predict_proba(X_A)[:, 1]
probs_B = xgb_B.predict_proba(X_B)[:, 1]

print(f'Set A predictions: {len(probs_A):,} rows')
print(f'Set B predictions: {len(probs_B):,} rows')

In [ ]:
# Bootstrap CIs for each hospital
results = []
for name, y_true, y_prob in [('Hospital A', y_A, probs_A), ('Hospital B', y_B, probs_B)]:
    auprc_mean, auprc_lo, auprc_hi   = bootstrap_ci(y_true, y_prob, metric='auprc')
    aucroc_mean, aucroc_lo, aucroc_hi = bootstrap_ci(y_true, y_prob, metric='auc_roc')
    print(f'{name}: AUPRC={auprc_mean:.4f} [{auprc_lo:.4f}, {auprc_hi:.4f}] | '
          f'AUC-ROC={aucroc_mean:.4f} [{aucroc_lo:.4f}, {aucroc_hi:.4f}]')
    results.append({
        'hospital': name,
        'auprc': auprc_mean, 'auprc_ci_lo': auprc_lo, 'auprc_ci_hi': auprc_hi,
        'auc_roc': aucroc_mean, 'auc_roc_ci_lo': aucroc_lo, 'auc_roc_ci_hi': aucroc_hi,
    })

res_df = pd.DataFrame(results)
res_df.to_csv(f'{METRICS_DIR}hospital_generalizability.csv', index=False)
print(f'Saved → {METRICS_DIR}hospital_generalizability.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#2196F3', '#4CAF50']

for ax, metric, label in zip(axes, ['auprc', 'auc_roc'], ['AUPRC', 'AUC-ROC']):
    for i, row in res_df.iterrows():
        err_lo = row[metric] - row[f'{metric}_ci_lo']
        err_hi = row[f'{metric}_ci_hi'] - row[metric]
        ax.bar(i, row[metric], color=colors[i], alpha=0.85, width=0.5,
               yerr=[[err_lo], [err_hi]], capsize=8, error_kw={'linewidth': 2})
        ax.text(i, row[metric] + err_hi + 0.003, f"{row[metric]:.4f}",
                ha='center', fontsize=10, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Hospital A
(Set A)', 'Hospital B
(Set B)'], fontsize=10)
    ax.set_ylabel(f'{label} (95% Bootstrap CI)')
    ax.set_title(f'{label} — XGBoost B by Hospital')
    ax.set_ylim(0, max(res_df[f'{metric}_ci_hi']) + 0.05)

plt.suptitle('Hospital Generalizability — XGBoost Strategy B (Best Model)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}hospital_generalizability.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {FIGURES_DIR}hospital_generalizability.png')

## Generalizability: Findings

- **AUC-ROC** is nearly identical across sites (Set A: 0.8504, Set B: 0.8391) , the model's discriminative ability generalises well
- **AUPRC** shows a modest gap (Set A: 0.1824, Set B: 0.1625) , likely attributable to differences in sepsis prevalence and lab measurement practices between hospital systems rather than a fundamental failure to generalise
- The consistent AUC-ROC across sites suggests the model has learned genuine clinical signal rather than hospital-specific artefacts
- The AUPRC gap is worth noting as a limitation , site-level prevalence differences affect the precision-recall trade-off independently of model quality